# Club 02 — Econometric Analysis in Stata

This notebook documents the **Stata commands used in Club 02** of the empirical analysis.

The notebook contains the original Stata commands together with Markdown explanations organized according to their econometric purpose.

## Empirical workflow

1. Panel-data setup and variable transformation
2. Descriptive statistics
3. Fixed-effects estimation
4. Normality diagnostics
5. Cross-sectional dependence
6. Slope heterogeneity
7. Second-generation panel unit-root testing
8. Panel cointegration testing
9. Method-of-Moments Quantile Regression
10. Quantile-regression visualization


## 1. Panel-data setup and variable transformation

This section establishes the country-year panel structure and creates the logarithmic form of multifactor productivity.

- `encode country, gen(country_num)` converts the country identifier into a numeric panel identifier.
- `xtset country_num year` declares the country-year panel structure.
- `gen log_mfp = log(mfp)` creates the natural logarithm of multifactor productivity.

The country is the cross-sectional dimension and year is the time dimension.

In [ ]:
* Panel-data identification
encode country, gen(country_num)

* Declare country-year panel structure
xtset country_num year

* Natural logarithm of multifactor productivity
gen log_mfp = log(mfp)

## 2. Descriptive statistics

The `summ` command provides descriptive statistics for the dependent variable and explanatory variables.

This step gives an initial overview of the distributions and scale of the variables before the econometric estimation.

In [ ]:
summ log_mfp fdi pci er fr pr gdpp

## 3. Baseline fixed-effects panel regression

The baseline model is estimated using the country fixed-effects estimator.

The `xtreg, fe` specification controls for unobserved, time-invariant country-specific characteristics.

The empirical model is:

$$
\ln(MFP_{it}) =
\alpha_i +
\beta_1 FDI_{it} +
\beta_2 PCI_{it} +
\beta_3 ER_{it} +
\beta_4 FR_{it} +
\beta_5 PR_{it} +
\beta_6 GDPP_{it} +
\varepsilon_{it}
$$

where $\alpha_i$ represents country-specific fixed effects.

The fixed-effects specification therefore focuses on within-country variation over time while controlling for time-invariant country characteristics.

In [ ]:
* Baseline fixed-effects panel regression
xtreg log_mfp fdi pci er fr pr gdpp, fe

## 4. Normality diagnostics

Two tests are used to examine whether the variables follow a normal distribution:

- **Shapiro–Wilk test (`swilk`)**
- **Shapiro–Francia test (`sfrancia`)**

The null hypothesis of these normality tests is that the variable is normally distributed.

These tests are used as distributional diagnostics for the variables included in the empirical analysis.

In [ ]:
* Shapiro-Wilk normality test
swilk log_mfp fdi pci er fr pr gdpp

* Shapiro-Francia normality test
sfrancia log_mfp fdi pci er fr pr gdpp

## 5. Cross-sectional dependence

Cross-sectional dependence occurs when observations for different countries are correlated.

This is an important diagnostic for international panel data because countries may be affected by common global or regional shocks.

The `xtcd` command tests the variables for cross-sectional dependence.

The variables are examined in two groups in the original analysis:

- `log_mfp`, `fdi`, and `pci`
- `er`, `fr`, `pr`, and `gdpp`

The results help determine whether cross-sectional dependence should be considered in subsequent panel-data procedures.

In [ ]:
* Cross-sectional dependence tests
xtcd log_mfp fdi pci

xtcd er fr pr gdpp

## 6. Slope heterogeneity

The `xthst` command tests whether the slope coefficients are homogeneous across countries.

The underlying question is whether the explanatory variables have the same effects for all countries or whether the magnitude of the relationships varies across cross-sectional units.

The null hypothesis can be expressed generally as:

$$
H_0: \beta_i = \beta
$$

against the alternative that the coefficients differ across countries.

Evidence of slope heterogeneity provides justification for using estimation approaches that allow relationships to vary across the conditional distribution or across panel units.

In [ ]:
* Test for slope heterogeneity
xthst log_mfp fdi pci er fr pr gdpp

## 7. Second-generation panel unit-root tests: CIPS

The `xtcips` command is used for **cross-sectionally augmented IPS (CIPS) panel unit-root testing**.

This is a second-generation panel unit-root procedure designed to account for cross-sectional dependence.

The general null hypothesis is that the panel series contains a unit root, while the alternative allows stationarity for at least some cross-sectional units.

The specification uses:

- `maxlags(1)` — maximum lag length of one;
- `bglags(1)` — one lag in the cross-sectional augmentation component.

### Variables tested in levels

The analysis tests:

- `log_mfp`
- `fdi`
- `pci`
- `er`
- `fr`
- `pr`
- `gdp`
- `gdpp`

### Variables tested in first differences

The original code additionally tests:

- `log_mfp`
- `pci`
- `gdpp`

using Stata's `d.` operator.

The first difference is:

$$
\Delta X_{it}=X_{it}-X_{i,t-1}
$$

The level and first-difference tests are used to establish the integration order of the variables before the cointegration analysis.

In [ ]:
* CIPS test for log MFP in levels
xtcips log_mfp, maxlags(1) bglags(1)

* CIPS test for first-differenced log MFP
xtcips d.log_mfp, maxlags(1) bglags(1)

* CIPS test for FDI
xtcips fdi, maxlags(1) bglags(1)

* CIPS test for PCI in levels
xtcips pci, maxlags(1) bglags(1)

* CIPS test for first-differenced PCI
xtcips d.pci, maxlags(1) bglags(1)

* CIPS test for ER
xtcips er, maxlags(1) bglags(1)

* CIPS test for FR
xtcips fr, maxlags(1) bglags(1)

* CIPS test for PR
xtcips pr, maxlags(1) bglags(1)

* CIPS test for GDP
xtcips gdp, maxlags(1) bglags(1)

* CIPS test for GDPP in levels
xtcips gdpp, maxlags(1) bglags(1)

* CIPS test for first-differenced GDPP
xtcips d.gdpp, maxlags(1) bglags(1)

## 8. Pedroni panel cointegration test

The `xtcointtest pedroni` command examines whether the variables share a long-run equilibrium relationship.

The test is conducted for:

- `log_mfp`
- `fdi`
- `pci`
- `er`
- `fr`
- `pr`
- `gdpp`

The general null hypothesis is:

$$
H_0: \text{No cointegration}
$$

Rejection of the null provides evidence that the variables are cointegrated and share a long-run relationship.

The Pedroni test is therefore used after investigating the stationarity properties of the panel variables.

In [ ]:
* Pedroni panel cointegration test
xtcointtest pedroni log_mfp fdi pci er fr pr gdpp

## 9. Method-of-Moments Quantile Regression (MMQR)

The `mmqreg` command estimates the relationship between multifactor productivity and the explanatory variables at different conditional quantiles.

The model is estimated at four points of the conditional distribution:

- 25th percentile ($\tau=0.25$)
- 50th percentile / median ($\tau=0.50$)
- 75th percentile ($\tau=0.75$)
- 90th percentile ($\tau=0.90$)

A general representation is:

$$
Q_{\ln(MFP_{it})}(\tau|X_{it})
=
\alpha_{\tau}
+
\beta_{1,\tau}FDI_{it}
+
\beta_{2,\tau}PCI_{it}
+
\beta_{3,\tau}ER_{it}
+
\beta_{4,\tau}FR_{it}
+
\beta_{5,\tau}PR_{it}
+
\beta_{6,\tau}GDPP_{it}
$$

The purpose is to determine whether the effects of the explanatory variables differ across low, middle, and high conditional levels of multifactor productivity.

This is particularly useful for identifying **heterogeneous effects** that may not be visible in a conventional mean regression.

In [ ]:
* MMQR at the 25th percentile
mmqreg log_mfp fdi pci er fr pr gdpp, q(25)

* MMQR at the 50th percentile (median)
mmqreg log_mfp fdi pci er fr pr gdpp, q(50)

* MMQR at the 75th percentile
mmqreg log_mfp fdi pci er fr pr gdpp, q(75)

* MMQR at the 90th percentile
mmqreg log_mfp fdi pci er fr pr gdpp, q(90)

## 10. Quantile-regression coefficient plot

The `qregplot` command is used to graphically display the estimated quantile-regression coefficients.

The plot makes it possible to examine how the estimated effects change across quantiles.

It can reveal:

- increasing or decreasing coefficient magnitudes;
- changes in the direction of relationships;
- differences between lower and higher productivity quantiles; and
- evidence of heterogeneous effects across the conditional distribution.

The graphical results complement the numerical MMQR estimates.

In [ ]:
* Plot quantile-regression coefficients
qregplot

## 11. Summary of the Club 02 methodology

The Club 02 empirical analysis follows the sequence:

**Panel setup → Variable transformation → Descriptive statistics → Fixed-effects regression → Normality diagnostics → Cross-sectional dependence → Slope heterogeneity → CIPS unit-root testing → Pedroni cointegration → MMQR → Quantile-regression plot**

The sequence begins with panel-data preparation and preliminary diagnostics, establishes the stationarity and long-run properties of the variables, and then examines heterogeneous relationships across different points of the conditional productivity distribution.